In [14]:
import datetime as dt
import openml
import numpy as np
import pandas as pd

## EDIT WORDING

Electricity dataset for Victoria, Aus between 1 January 2015 and 6 October 2020. 
https://www.kaggle.com/datasets/aramacus/electricity-demand-in-victoria-australia/data

date : datetime, the date of the recording  
demand : float, a total daily electricity demand in MWh  
RRP : float, a recommended retail price in AUD per MWh  
demand_pos_RRP : float, a total daily demand at positive RRP in MWh  
RRP_positive : float, an averaged positive RRP, weighted by the corresponding intraday demand in AUD per MWh  
demand_neg_RRP : float, an total daily demand at negative RRP in MWh  
RRP_negative : float, an average negative RRP, weighted by the corresponding intraday demand in AUD per MWh  
frac_at_neg_RRP : float, a fraction of the day when the demand was traded at negative RRP  
min_temperature : float, minimum temperature during the day in Celsius  
max_temperature : float, maximum temperature during the day in Celsius  
solar_exposure : float, total daily sunlight energy in MJ/m^2  
rainfall : float, daily rainfall in mm  
school_day : boolean, if students were at school on that day  
holiday : boolean, if the day was a state or national holiday  

In [16]:


filepath ='../data/raw/complete_dataset.csv'

df_elec = pd.read_csv(filepath, parse_dates = ['date'])
df_elec = df_elec.set_index('date')
df_elec.head()

,demand,RRP,demand_pos_RRP,RRP_positive,demand_neg_RRP,RRP_negative,frac_at_neg_RRP,min_temperature,max_temperature,solar_exposure,rainfall,school_day,holiday
date,,,,,,,,,,,,,
2015-01-01,99635.030,25.633696,97319.240,26.415953,2315.790,-7.240000,0.020833,13.3,26.9,23.6,0.0,N,Y
2015-01-02,129606.010,33.138988,121082.015,38.837661,8523.995,-47.809777,0.062500,15.4,38.8,26.8,0.0,N,N
2015-01-03,142300.540,34.564855,142300.540,34.564855,0.000,0.000000,0.000000,20.0,38.2,26.5,0.0,N,N
2015-01-04,104330.715,25.005560,104330.715,25.005560,0.000,0.000000,0.000000,16.3,21.4,25.2,4.2,N,N
2015-01-05,118132.200,26.724176,118132.200,26.724176,0.000,0.000000,0.000000,15.0,22.0,30.7,0.0,N,N


## Cleaning Data

In [17]:
# Check for missing dates in the range 1 January 2015 to 6 October 2020. 
full_range = pd.date_range(start=df_elec.index.min(), end=df_elec.index.max(), freq='D')
missing_dates = full_range.difference(df_elec.index)
missing_dates

DatetimeIndex([], dtype='datetime64[ns]', freq='D')

In [18]:
# Checking for missing values
df_elec.isnull().sum()

demand             0
RRP                0
demand_pos_RRP     0
RRP_positive       0
demand_neg_RRP     0
RRP_negative       0
frac_at_neg_RRP    0
min_temperature    0
max_temperature    0
solar_exposure     1
rainfall           3
school_day         0
holiday            0
dtype: int64

In [19]:
# Because solar_exposure and rainfall vary continuously, we apply quadratic interpolation to fill the missing vals
df_elec['rainfall'] = df_elec['rainfall'].interpolate(method='polynomial', order=2)
df_elec['solar_exposure'] = df_elec['solar_exposure'].interpolate(method='polynomial', order=2)

In [23]:
df_elec.describe()

,demand,RRP,demand_pos_RRP,RRP_positive,demand_neg_RRP,RRP_negative,frac_at_neg_RRP,min_temperature,max_temperature,solar_exposure,rainfall
count,2106.000000,2106.000000,2106.000000,2106.000000,2106.000000,2106.000000,2106.000000,2106.000000,2106.000000,2106.000000,2106.000000
mean,120035.476503,76.079554,119252.305055,76.553847,783.171448,-2.686052,0.008547,11.582289,20.413200,14.744525,1.503320
std,13747.993761,130.246805,14818.631319,130.114184,3578.920686,19.485432,0.039963,4.313711,6.288693,7.943815,4.305425
min,85094.375000,-6.076028,41988.240000,13.568986,0.000000,-342.220000,0.000000,0.600000,9.000000,0.700000,-1.008289
25%,109963.650000,38.707040,109246.250000,39.117361,0.000000,0.000000,0.000000,8.500000,15.525000,8.200000,0.000000
50%,119585.912500,66.596738,119148.082500,66.869058,0.000000,0.000000,0.000000,11.300000,19.100000,12.750000,0.000000
75%,130436.006250,95.075012,130119.477500,95.130181,0.000000,0.000000,0.000000,14.600000,23.900000,20.700000,0.800000
max,170653.840000,4549.645105,170653.840000,4549.645105,57597.595000,0.000000,0.625000,28.000000,43.500000,33.300000,54.600000


## Creating Features

### Creating Date Features

In [6]:
# Convert cols school_day and holiday to booleans
for col in ['school_day', 'holiday']:
    df_elec[col] = df_elec[col].map(lambda x: True if x=='Y' else False)

# Extract the day of the week
df_elec['day'] = df_elec.index.strftime('%a')

# Extract the month
df_elec['month'] = df_elec.index.strftime('%b')

# Create col is_weekend
df_elec['weekend'] = (df_elec.index.weekday >= 5)

In [7]:
df_elec.head()

,demand,RRP,demand_pos_RRP,RRP_positive,demand_neg_RRP,RRP_negative,frac_at_neg_RRP,min_temperature,max_temperature,solar_exposure,rainfall,school_day,holiday,day,weekend
date,,,,,,,,,,,,,,,
2015-01-01,99635.030,25.633696,97319.240,26.415953,2315.790,-7.240000,0.020833,13.3,26.9,23.6,0.0,False,True,Thu,False
2015-01-02,129606.010,33.138988,121082.015,38.837661,8523.995,-47.809777,0.062500,15.4,38.8,26.8,0.0,False,False,Fri,False
2015-01-03,142300.540,34.564855,142300.540,34.564855,0.000,0.000000,0.000000,20.0,38.2,26.5,0.0,False,False,Sat,True
2015-01-04,104330.715,25.005560,104330.715,25.005560,0.000,0.000000,0.000000,16.3,21.4,25.2,4.2,False,False,Sun,True
2015-01-05,118132.200,26.724176,118132.200,26.724176,0.000,0.000000,0.000000,15.0,22.0,30.7,0.0,False,False,Mon,False


### Creating Feature for Hazelwood Power Station Closure
In March 2017, the Hazelwood Power Station was closed. Prior to its closure, it had been responsible for ~22% of Victoria's electricity.
The electricity unit shut down took place between March 27 and March 29. 

In [8]:
closure = pd.Timestamp('2017-03-30')
df_elec['hazelwood_closed'] = (df_elec.index >= closure)

### Creating Features for COV-19

In [9]:
# Melbourne lockdown dates taken from https://www.publish.csiro.au/ma/pdf/MA22002
mel_lockdowns = [
    (pd.Timestamp('2020-03-31'), pd.Timestamp('2020-05-13')),
    (pd.Timestamp('2020-07-09'), pd.Timestamp('2020-10-28')),
    (pd.Timestamp('2021-02-12'), pd.Timestamp('2021-02-17')),
    (pd.Timestamp('2021-05-28'), pd.Timestamp('2021-06-09')),
    (pd.Timestamp('2021-07-16'), pd.Timestamp('2021-07-28')),
    (pd.Timestamp('2021-08-5'), pd.Timestamp('2021-10-23'))
]

df_elec['mel_lockdown'] = False

for start, end in mel_lockdowns:
    df_elec['mel_lockdown'] |= df_elec.index.to_series().between(start, end)

df_elec.head()

,demand,RRP,demand_pos_RRP,RRP_positive,demand_neg_RRP,RRP_negative,frac_at_neg_RRP,min_temperature,max_temperature,solar_exposure,rainfall,school_day,holiday,day,weekend,hazelwood_closed,mel_lockdown
date,,,,,,,,,,,,,,,,,
2015-01-01,99635.030,25.633696,97319.240,26.415953,2315.790,-7.240000,0.020833,13.3,26.9,23.6,0.0,False,True,Thu,False,False,False
2015-01-02,129606.010,33.138988,121082.015,38.837661,8523.995,-47.809777,0.062500,15.4,38.8,26.8,0.0,False,False,Fri,False,False,False
2015-01-03,142300.540,34.564855,142300.540,34.564855,0.000,0.000000,0.000000,20.0,38.2,26.5,0.0,False,False,Sat,True,False,False
2015-01-04,104330.715,25.005560,104330.715,25.005560,0.000,0.000000,0.000000,16.3,21.4,25.2,4.2,False,False,Sun,True,False,False
2015-01-05,118132.200,26.724176,118132.200,26.724176,0.000,0.000000,0.000000,15.0,22.0,30.7,0.0,False,False,Mon,False,False,False


## EDA

### Plot of Electricity Demand Over Time